## GradeCAM++

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from torchvision import models, transforms

# --- Paths ---
BASE_DIR = Path.cwd().resolve()
DATA_DIR = BASE_DIR / "isic-2024"
OUTPUT_DIR = BASE_DIR / "heatmap"
MODEL_PATH = BASE_DIR / "shufflenetv2_skin_cancer.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ["benign", "malignant"] # Index 0: Non-cancer, Index 1: Cancer

# --- Model Loader ---
def load_model(model_path: Path, num_classes: int = 2):
    model = models.shufflenet_v2_x1_0(weights=None, num_classes=num_classes)
    if model_path.exists():
        try:
            checkpoint = torch.load(str(model_path), map_location="cpu")
            state_dict = checkpoint.get("state_dict", checkpoint)
            
            model_state = model.state_dict()
            compatible_state = {}
            for k, v in state_dict.items():
                if k in model_state and model_state[k].shape == v.shape:
                    compatible_state[k] = v
            
            model.load_state_dict(compatible_state, strict=False)
            print(f"Model loaded from {model_path}")
        except Exception as e:
            print(f"Warning: Could not load model weights: {e}")
            print(f"Using untrained ShuffleNetV2")
    else:
        print(f"Model file not found at {model_path}, using untrained ShuffleNetV2")
    model.eval()
    return model

# --- Preprocessing ---
def preprocess_image(img_path: str):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    
    # FIX 1: Convert BGR to RGB before passing to PyTorch
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    return transform(img_resized).unsqueeze(0)

# --- Prediction ---
def predict(model, img_tensor):
    with torch.no_grad(): # Optimization for inference
        preds = model(img_tensor)
    _, idx = torch.max(preds, 1)
    return idx.item(), CLASS_NAMES[idx.item()]

# --- Grad-CAM++ ---
def get_conv_layer(model, layer_name="stage4.0"):
    for name, layer in model.named_modules():
        if name == layer_name:
            return layer
    raise ValueError(f"Layer '{layer_name}' not found.")

def compute_gradcam_pp(model, img_tensor, class_idx, conv_layer_name="stage4.0"):
    conv_layer = get_conv_layer(model, conv_layer_name)
    activations, gradients = None, None

    def fwd_hook(_, __, output): 
        nonlocal activations
        activations = output
        
    def bwd_hook(_, __, grad_output): 
        nonlocal gradients
        gradients = grad_output[0]

    f_hook = conv_layer.register_forward_hook(fwd_hook)
    # FIX 2: Use stable register_full_backward_hook
    b_hook = conv_layer.register_full_backward_hook(bwd_hook)

    # Enable gradients just for tracking back to layer
    img_tensor.requires_grad_() 
    preds = model(img_tensor)
    score = preds[0, class_idx]
    
    model.zero_grad()
    score.backward(retain_graph=True)

    f_hook.remove()
    b_hook.remove()

    A = activations.detach().cpu().numpy()[0]
    dY_dA = gradients.detach().cpu().numpy()[0]

    # FIX 3: Robust shape handling for Grad-CAM++ equations
    dY_dA2 = dY_dA ** 2
    dY_dA3 = dY_dA ** 3
    sum_A = np.sum(A, axis=(1, 2), keepdims=True)
    
    denom = 2 * dY_dA2 + sum_A * dY_dA3
    alpha = np.where(denom != 0, dY_dA2 / denom, 0)

    # Calculate channel-wise weightings safely
    w_k = np.sum(alpha * np.maximum(dY_dA, 0), axis=(1, 2))
    w_k = w_k[:, np.newaxis, np.newaxis] # Explicit 3D broadcast matrix
    
    heatmap = np.sum(w_k * A, axis=0)
    heatmap = np.maximum(heatmap, 0)
    
    max_val = np.max(heatmap)
    if max_val != 0:
        heatmap /= max_val
        
    return heatmap

# --- Overlay ---
def overlay_heatmap(img_path, heatmap, alpha=0.4, class_idx=None):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    output = cv2.addWeighted(img, alpha, heatmap, 1 - alpha, 0)
    
    if class_idx is not None:
        cv2.putText(output, str(class_idx), (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return output

# --- Run Example ---
model = load_model(MODEL_PATH)
img_path = str(DATA_DIR / "clean_ISIC_0190307.jpg")

img_tensor = preprocess_image(img_path)
class_idx, class_name = predict(model, img_tensor)
print(f"Prediction: {class_name} (index {class_idx})")

# Re-run forward step with gradient tracing for map extraction
heatmap = compute_gradcam_pp(model, img_tensor, class_idx, conv_layer_name="stage4.0")
output_img = overlay_heatmap(img_path, heatmap, class_idx=class_idx)

out_path = str(OUTPUT_DIR / "clean_ISIC_0190307_gradcam.jpg")
cv2.imwrite(out_path, output_img)
print(f"Grad-CAM++ saved at {out_path}")


Model loaded from D:\CNN-Models\promed\shufflenetv2_skin_cancer.pth
Prediction: MALIGNANT
Probabilities -> Benign: 0.20%, Malignant: 99.80%
Grad-CAM++ saved at D:\CNN-Models\promed\heatmap\clean_ISIC_0190307_gradcam.jpg
